In [39]:
from pyspark.sql import functions as F

BRONZE = "business_bronze"
SILVER = "business_silver"

df_business = spark.table(BRONZE)


StatementMeta(, e740649c-51a5-487a-b7c4-e2e231ff6721, 41, Finished, Available, Finished, False)

In [33]:
base_cols = [
    "business_id","name","address","city","state","postal_code",
    "latitude","longitude","stars","review_count","is_open","categories",
    "_ingest_ts","_source_file","_batch_id"
]

df_business_silver = df_business.select(*base_cols)

StatementMeta(, e740649c-51a5-487a-b7c4-e2e231ff6721, 35, Finished, Available, Finished, False)

In [34]:
df_business_silver = (df_business_silver
    .withColumn("latitude", F.col("latitude").cast("double"))
    .withColumn("longitude", F.col("longitude").cast("double"))
    .withColumn("stars", F.col("stars").cast("double"))
    .withColumn("review_count", F.col("review_count").cast("int"))
    .withColumn("is_open", F.col("is_open").cast("int"))

    .dropDuplicates(["business_id"])

    .filter(F.col("business_id").isNotNull())
    .filter(F.col("stars").isNotNull())
    .filter((F.col("stars")>=0)&(F.col("stars")<=5))
    .filter(F.col("review_count").isNotNull())
    .filter(F.col("review_count")>=0)
    .filter(F.col("is_open").isNotNull())
    .filter(F.col("is_open").isin(0, 1))
)

StatementMeta(, e740649c-51a5-487a-b7c4-e2e231ff6721, 36, Finished, Available, Finished, False)

In [35]:
(df_business_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER)
)

StatementMeta(, e740649c-51a5-487a-b7c4-e2e231ff6721, 37, Finished, Available, Finished, False)

In [36]:
df_business_silver.printSchema()

StatementMeta(, e740649c-51a5-487a-b7c4-e2e231ff6721, 38, Finished, Available, Finished, False)

root
 |-- business_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- stars: double (nullable = true)
 |-- review_count: integer (nullable = true)
 |-- is_open: integer (nullable = true)
 |-- categories: string (nullable = true)
 |-- _ingest_ts: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _batch_id: string (nullable = true)



In [37]:
print("business_silver rows: ", df_business_silver.count())

StatementMeta(, e740649c-51a5-487a-b7c4-e2e231ff6721, 39, Finished, Available, Finished, False)

business_silver rows:  150346


In [38]:
mssparkutils.notebook.exit("SUCCESS")

StatementMeta(, e740649c-51a5-487a-b7c4-e2e231ff6721, 40, Finished, Available, Finished, False)

ExitValue: SUCCESS

# Silver Layer Contract (Frozen Semantics)

This document defines the **frozen semantic contract** for the Silver layer.
Silver is the project’s **stable, query-friendly, business-semantic layer**.
All downstream (Gold, analytics, BI) depends on this contract.

---

## Global Conventions

### Naming
- All column names are lowercase and snake_case.
- ID columns use `*_id`.
- Timestamps use `*_ts` (Spark timestamp).
- Dates use `*_date` (Spark date).

### Data Quality (Baseline)
- Primary keys are unique per table.
- Fact tables are partitioned by date.
- Silver tables are written as Delta tables.
- Lineage column `_ingest_ts` is preserved when available.

---

## Tables

---

## 1) dbo.review_silver (Fact Table)

**Purpose**  
Fact table representing review events.

**Grain**  
- 1 row = 1 review.

**Primary Key**  
- `review_id`

**Foreign Keys**
- `business_id` → dbo.business_silver
- `user_id` → dbo.user_silver

**Partition**
- `review_date`

**Schema (Frozen)**
- `review_id` (string)
- `business_id` (string)
- `user_id` (string)
- `stars` (double)
- `useful` (long)
- `funny` (long)
- `cool` (long)
- `text` (string)
- `_ingest_ts` (timestamp)
- `review_ts` (timestamp)
- `review_date` (date)

**Typical Query Patterns**
- Filter by `review_date`
- Join on `business_id`, `user_id`
- Aggregate by `stars`, `useful`, `funny`, `cool`

**Source**
- dbo.review_bronze

---

## 2) dbo.business_silver (Dimension Table)

**Purpose**  
Business entity dimension.

**Grain**  
- 1 row = 1 business.

**Primary Key**
- `business_id`

**Partition**
- None

**Schema (Frozen)**
- `business_id` (string)
- `name` (string)
- `address` (string)
- `city` (string)
- `state` (string)
- `postal_code` (string)
- `latitude` (double)
- `longitude` (double)
- `stars` (double)
- `review_count` (integer)
- `is_open` (integer)
- `categories` (string)
- `_ingest_ts` (timestamp)

**Typical Query Patterns**
- Filter by `city`, `state`
- Filter by `is_open`
- Aggregate by `stars`, `review_count`

**Source**
- dbo.business_bronze

---

## 3) dbo.user_silver (Dimension Table)

**Purpose**  
User entity dimension.

**Grain**  
- 1 row = 1 user.

**Primary Key**
- `user_id`

**Partition**
- None

**Schema (Frozen)**
- `user_id` (string)
- `name` (string)
- `review_count` (integer)
- `average_stars` (double)
- `fans` (integer)
- `_ingest_ts` (timestamp)
- `_source_file` (string)

**Typical Query Patterns**
- Filter by `review_count`
- Filter by `average_stars`
- Ranking by `fans`

**Source**
- dbo.user_bronze

---

# Schema Evolution Policy (Silver)

Silver is a **contract layer**. Changes must follow these rules.

## Safe / Backward-Compatible Changes
- Add new nullable columns.
- Add new columns with safe defaults.
- Backfill data via overwrite or merge.

## Breaking Changes (Require Versioning)
- Rename existing columns.
- Change column data types.
- Change table grain.
- Remove existing columns.
- Change partition columns.

## Versioning Strategy
- Breaking changes must create a new table:
  - e.g. `dbo.review_silver_v2`
- Keep old and new versions temporarily.
- Migrate downstream consumers before deprecating v1.
